# ✂️ SAM3 — Bounding Box Refinement

> Mevcut kaba YOLO etiketleri → SAM3 box prompt → Hassas mask → Tight bbox

## Ne Yapar?
Elindeki dataset'teki YOLO `.txt` etiketlerini SAM3 ile piksel-hassasiyetinde iyileştirir.
Sınıf ID'leri değişmez — sadece bounding box koordinatları daha sıkı hale gelir.

## Kaggle Inputs
| Slot | Açıklama |
|------|----------|
| Dataset | Görüntüler + mevcut YOLO `.txt` etiketleri |

## Kaggle Secrets
| Key | Value |
|-----|-------|
| `HF_TOKEN` | HuggingFace Read token |

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────────────────
!pip install -q "transformers==5.5.0" accelerate pillow

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 2. HuggingFace Token ─────────────────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('✅ HF_TOKEN alındı')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN eksik! Add-ons → Secrets → HF_TOKEN ekle')

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK ✅')

In [ ]:
# ── 3. Konfigürasyon ─────────────────────────────────────────────────────────
from pathlib import Path

# Sınıf haritası — dataset'indeki class_id → isim eşlemesi
# data.yaml'dan alarak güncelle
CLASS_MAP = {
    0: 'tank',
    1: 'armored vehicle',
    2: 'military truck',
    3: 'helicopter',
    4: 'soldier',
    5: 'drone',
}

SAM3_MODEL_ID      = 'facebook/sam3'

MIN_SCORE          = 0.5    # SAM3 mask kalite eşiği
EXPAND_RATIO       = 0.05   # Box prompt'u %5 büyüt (kenar kaçırmayı önler)

IMAGE_INPUT_DIR    = ''     # boş = otomatik bul
LABEL_INPUT_DIR    = ''     # boş = görüntülerle aynı dizinde ara
OUTPUT_DIR         = Path('/kaggle/working/refined')
IMAGE_EXTENSIONS   = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

print('Config OK ✅')

In [ ]:
# ── 4. SAM3 Yükle ───────────────────────────────────────────────────────────
from transformers import Sam3Model, Sam3Processor

print(f'SAM3 yükleniyor...')
sam_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN)
sam_model     = Sam3Model.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN).to(DEVICE).eval()
print(f'SAM3 hazır ✅  ({sum(p.numel() for p in sam_model.parameters())/1e6:.0f}M param)')

In [ ]:
# ── 5. Görüntü + Etiket Çiftlerini Bul ──────────────────────────────────────
import os
from PIL import Image
import numpy as np

def find_image_dir(base='/kaggle/input'):
    best_dir, best_count = Path(base), 0
    for dirpath, _, filenames in os.walk(base):
        imgs = [f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) > best_count:
            best_count, best_dir = len(imgs), Path(dirpath)
    return best_dir

img_dir = Path(IMAGE_INPUT_DIR) if IMAGE_INPUT_DIR else find_image_dir()
all_images = sorted(p for p in img_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)

# Her görüntü için etiket dosyasını bul
pairs = []
for img_path in all_images:
    # Önce aynı klasörde ara, sonra labels/ klasöründe
    candidates = [
        img_path.with_suffix('.txt'),
        img_path.parent.parent / 'labels' / (img_path.stem + '.txt'),
        img_path.parent / 'labels' / (img_path.stem + '.txt'),
    ]
    lbl = next((p for p in candidates if p.exists()), None)
    if lbl and lbl.stat().st_size > 0:
        pairs.append((img_path, lbl))

print(f'📂 Görüntü dizini    : {img_dir}')
print(f'🖼️  Toplam görüntü    : {len(all_images):,}')
print(f'🏷️  Etiketli görüntü  : {len(pairs):,}')
if pairs:
    print(f'\nÖrnek etiket: {pairs[0][1].read_text().strip()[:100]}')

In [ ]:
# ── 6. Çıktı Klasörleri ──────────────────────────────────────────────────────
img_out   = OUTPUT_DIR / 'images'
label_out = OUTPUT_DIR / 'labels'
img_out.mkdir(parents=True, exist_ok=True)
label_out.mkdir(parents=True, exist_ok=True)
print(f'Çıktı: {OUTPUT_DIR}')

In [ ]:
# ── 7. SAM3 Box Refinement Fonksiyonu ────────────────────────────────────────

def expand_box(x1, y1, x2, y2, W, H, ratio=EXPAND_RATIO):
    """Box'u biraz büyüt — nesnenin kenarını kaçırmamak için."""
    dx = (x2 - x1) * ratio
    dy = (y2 - y1) * ratio
    return (
        max(0, x1 - dx), max(0, y1 - dy),
        min(W, x2 + dx), min(H, y2 + dy)
    )

def mask_to_bbox(mask):
    """Boolean mask → [x1, y1, x2, y2] sıkı bbox."""
    rows = mask.any(axis=1)
    cols = mask.any(axis=0)
    if not rows.any():
        return None
    y1 = int(rows.argmax())
    y2 = int(len(rows) - rows[::-1].argmax() - 1)
    x1 = int(cols.argmax())
    x2 = int(len(cols) - cols[::-1].argmax() - 1)
    return x1, y1, x2, y2

def refine_bbox(image: Image.Image, class_id: int,
                cx: float, cy: float, bw: float, bh: float):
    """
    Mevcut YOLO bbox'ı SAM3 ile iyileştir.
    Döner: (cx, cy, bw, bh) normalized — başarısızsa orijinali döndürür.
    """
    W, H = image.size
    class_name = CLASS_MAP.get(class_id, f'object {class_id}')

    # YOLO normalized → pixel XYXY
    px1 = (cx - bw / 2) * W
    py1 = (cy - bh / 2) * H
    px2 = (cx + bw / 2) * W
    py2 = (cy + bh / 2) * H

    # Box prompt'u biraz genişlet
    ex1, ey1, ex2, ey2 = expand_box(px1, py1, px2, py2, W, H)

    try:
        inputs = sam_processor(
            images=image,
            text=class_name,                          # hangi nesneyi aradığımız
            input_boxes=[[[ex1, ey1, ex2, ey2]]],     # nerede arayacağımız
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = sam_model(**inputs)

        # SAM3 genellikle 3 mask tahmini döndürür — en yüksek skorluyu al
        masks  = sam_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs['original_sizes'].cpu(),
            inputs['reshaped_input_sizes'].cpu()
        )
        scores = outputs.iou_scores[0, 0].cpu()  # (3,)

        best_idx = int(scores.argmax())
        if float(scores[best_idx]) < MIN_SCORE:
            return cx, cy, bw, bh  # düşük güven → orijinali koru

        mask = masks[0][0][best_idx].numpy().astype(bool)  # (H, W)
        bbox = mask_to_bbox(mask)
        if bbox is None:
            return cx, cy, bw, bh

        nx1, ny1, nx2, ny2 = bbox
        ncx = ((nx1 + nx2) / 2) / W
        ncy = ((ny1 + ny2) / 2) / H
        nbw = (nx2 - nx1) / W
        nbh = (ny2 - ny1) / H
        return (
            max(0., min(1., ncx)),
            max(0., min(1., ncy)),
            max(0.001, min(1., nbw)),
            max(0.001, min(1., nbh)),
        )

    except Exception as e:
        return cx, cy, bw, bh  # hata → orijinali koru

print('Refinement fonksiyonu hazır ✅')

In [ ]:
# ── 8. Ana Döngü ─────────────────────────────────────────────────────────────
import shutil
from collections import defaultdict

stats = {
    'total'   : len(pairs),
    'refined' : 0,
    'kept'    : 0,   # orijinal korunan
    'errors'  : 0,
    'total_ann': 0,
}

print(f'🚀 {len(pairs):,} görüntü işleniyor (SAM3 box refinement)')

for i, (img_path, lbl_path) in enumerate(pairs):
    try:
        image   = Image.open(img_path).convert('RGB')
        W, H    = image.size
        lines_in  = lbl_path.read_text().strip().splitlines()
        lines_out = []
        img_refined = 0

        for line in lines_in:
            parts = line.strip().split()
            if len(parts) < 5:
                lines_out.append(line)
                continue

            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])

            ncx, ncy, nbw, nbh = refine_bbox(image, cls_id, cx, cy, bw, bh)

            # Değişim oldu mu?
            changed = abs(ncx-cx) + abs(ncy-cy) + abs(nbw-bw) + abs(nbh-bh) > 0.001
            if changed:
                img_refined += 1
            else:
                stats['kept'] += 1

            lines_out.append(f'{cls_id} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}')
            stats['total_ann'] += 1

        # Görüntü + yeni label'ı kaydet
        shutil.copy2(img_path, img_out / img_path.name)
        (label_out / (img_path.stem + '.txt')).write_text('\n'.join(lines_out))

        if img_refined > 0:
            stats['refined'] += 1

    except Exception as e:
        stats['errors'] += 1
        print(f'  ❌ [{img_path.name}]: {e}')

    if (i+1) % 10 == 0 or i == len(pairs)-1:
        pct = (i+1) / len(pairs) * 100
        print(f'  [{i+1:>4}/{len(pairs)}] {pct:5.1f}%  '
              f'refined={stats["refined"]}  kept={stats["kept"]}  errors={stats["errors"]}')

print('\n✅ Tamamlandı!')

In [ ]:
# ── 9. İstatistikler ─────────────────────────────────────────────────────────
print('='*50)
print('📊 REFINEMENT SONUÇLARI')
print('='*50)
print(f'Toplam görüntü        : {stats["total"]:,}')
print(f'En az 1 bbox iyileşti : {stats["refined"]:,}')
print(f'Toplam annotation     : {stats["total_ann"]:,}')
print(f'Hata                  : {stats["errors"]:,}')
print('='*50)

In [ ]:
# ── 10. Karşılaştırmalı Görselleştirme ───────────────────────────────────────
import random, matplotlib.pyplot as plt, matplotlib.patches as patches

PALETTE = ['#00e676','#40c4ff','#ff6d00','#ea80fc','#ffd740','#69f0ae','#ff4081','#80d8ff']

def draw_boxes(ax, img_path, lbl_path, title):
    img = Image.open(img_path)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=8)
    if not lbl_path or not lbl_path.exists(): return
    W, H = img.size
    for line in lbl_path.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cid = int(p[0]); cx,cy,bw,bh = map(float,p[1:5])
        x1,y1 = (cx-bw/2)*W, (cy-bh/2)*H
        c = PALETTE[cid % len(PALETTE)]
        ax.add_patch(patches.Rectangle((x1,y1),bw*W,bh*H,lw=2,edgecolor=c,facecolor='none'))
        ax.text(x1+2,y1-4,CLASS_MAP.get(cid,f'cls{cid}'),color=c,fontsize=7,
                bbox=dict(boxstyle='square,pad=0.1',fc='black',alpha=0.75,lw=0))

# Rastgele 6 örnek — ÖNCE vs SONRA karşılaştırması
sample = random.sample(pairs, min(6, len(pairs)))
fig, axes = plt.subplots(len(sample), 2, figsize=(10, len(sample)*3.5))
if len(sample) == 1: axes = [axes]

for row, (img_path, orig_lbl) in enumerate(sample):
    new_lbl = label_out / (img_path.stem + '.txt')
    draw_boxes(axes[row][0], img_path, orig_lbl, f'ÖNCE — {img_path.name[:30]}')
    draw_boxes(axes[row][1], img_out / img_path.name, new_lbl, 'SONRA (SAM3 refined)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('comparison.png kaydedildi')

In [ ]:
# ── 11. data.yaml + ZIP ──────────────────────────────────────────────────────
import yaml, zipfile, time

yaml_content = {
    'path' : str(OUTPUT_DIR),
    'train': 'images',
    'val'  : 'images',
    'nc'   : len(CLASS_MAP),
    'names': list(CLASS_MAP.values()),
}
with open(OUTPUT_DIR/'data.yaml','w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

zip_path = Path('/kaggle/working') / f'sam3_refined_{int(time.time())}.zip'
files    = [f for f in OUTPUT_DIR.rglob('*') if f.is_file()]
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as zf:
    for f in files: zf.write(f, f.relative_to(OUTPUT_DIR))

print(f'✅ {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
print('→ Kaggle Output → indir → unilabellm\'e yükle')